In [71]:
# --- Cell 1: 环境初始化 ---
%run config.ipynb

import tushare as ts
import pandas as pd
import time
import logging
from datetime import datetime, timedelta, timezone

# 日志配置
log_file = LOG_DIR / f"sync_job_{datetime.now().strftime('%Y%m%d')}.log"
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - [%(levelname)s] - %(message)s',
    handlers=[logging.FileHandler(log_file, encoding='utf-8'), logging.StreamHandler()]
)

ts.set_token(TUSHARE_TOKEN)
pro = ts.pro_api()

# 确定目标日期
tz_china = timezone(timedelta(hours=8))
china_now = datetime.now(tz_china)
TARGET_DT = china_now.strftime('%Y%m%d') if china_now.hour >= 16 else (china_now - timedelta(days=1)).strftime('%Y%m%d')
logging.info(f"✅ 环境加载成功！目标截止日期: {TARGET_DT}")


2026-05-15 12:11:37,155 - [INFO] - ✅ 环境加载成功！目标截止日期: 20260514


In [72]:
# --- Cell 2: 全量/静态数据拉取引擎 (适用于无需按天循环的表) ---
def sync_full_table(table_key, api_name, fields):
    logging.info(f"📦 全量同步静态表: {table_key}")
    file_path = OUT_DIR / f"{table_key}.xlsx"
    try:
        df = getattr(pro, api_name)(fields=fields)
        if not df.empty:
            df.to_excel(file_path, index=False)
            logging.info(f"✅ {table_key} 同步完成，共 {len(df)} 行")
    except Exception as e:
        logging.error(f"❌ {table_key} 拉取失败: {str(e)}")


In [73]:
# --- Cell 3. 动态增量同步引擎 (支持所有日历类表的断点续传) ---
# 支持季拆分与单文件
def sync_dynamic_incremental(table_key, api_name, date_param, fields, split_mode, extra_params=None):
    logging.info(f"🚀 启动日期增量拉取: {table_key}")
    cal = pro.trade_cal(exchange='SSE', start_date=START_DATE, end_date=TARGET_DT)
    trade_days = cal.loc[cal['is_open'] == 1, 'cal_date'].tolist()
    
    file_groups = {}
    if split_mode == 'single_file':
        file_groups[table_key] = trade_days
    else:
        for d in trade_days:
            y, m = d[:4], int(d[4:6])
            q = (m - 1) // 3 + 1
            key = f"{table_key}_{y}_quarter{q}"
            file_groups.setdefault(key, []).append(d)

    for file_name_base, days_in_group in file_groups.items():
        file_path = OUT_DIR / f"{file_name_base}.xlsx"
        existing_df = pd.DataFrame()
        needed_days = days_in_group
        
        if file_path.exists():
            try:
                existing_df = pd.read_excel(file_path, dtype={date_param: str})
                if not existing_df.empty:
                    last_dt = existing_df[date_param].max()
                    needed_days = [d for d in days_in_group if d > last_dt]
                if not needed_days: continue
            except: pass

        new_shards = []
        for d in needed_days:
            try:
                query_params = {date_param: d, 'fields': fields}
                if extra_params: query_params.update(extra_params)
                df_d = getattr(pro, api_name)(**query_params)
                if not df_d.empty: new_shards.append(df_d)
                time.sleep(SLEEP_TIME) 
            except Exception as e:
                logging.error(f"❌ {table_key} @ {d} 错误: {e}")
        
        if new_shards:
            final_df = pd.concat([existing_df] + new_shards, ignore_index=True)
            
            # 智能去重逻辑：如果有 ts_code 就执行去重
            if 'ts_code' in final_df.columns:
                # 动态组装用来去重的列
                drop_cols = ['ts_code']
                if date_param in final_df.columns:
                    drop_cols.append(date_param)      # 常规行情表用 trade_date 去重
                elif 'end_date' in final_df.columns:
                    drop_cols.append('end_date')      # 像 11 表这种没有 date_param 的，用 end_date 去重
                    
                # 执行去重
                final_df.drop_duplicates(subset=drop_cols, inplace=True)
                
            final_df.to_excel(file_path, index=False)

In [ ]:
# --- Cell 4: 财务报表引擎 (按股票代码分块防爆版) ---
def sync_by_stock_logic(table_key, api_name, fields):
    logging.info(f"🔄 启动按股票循环拉取: {table_key}")
    
    # 1. 获取股票列表
    stock_basic_path = OUT_DIR / "01_stock_basic.xlsx"
    if not stock_basic_path.exists():
        logging.error("❌ 找不到 01_stock_basic.xlsx，请先运行该接口")
        return
    
    all_stocks = pd.read_excel(stock_basic_path)['ts_code'].tolist()
    
    # 2. 核心升级：按 1000 只股票切片，防止撑爆 Excel
    CHUNK_SIZE = 1000 
    total_chunks = (len(all_stocks) - 1) // CHUNK_SIZE + 1
    
    for chunk_idx in range(total_chunks):
        start_idx = chunk_idx * CHUNK_SIZE
        end_idx = min((chunk_idx + 1) * CHUNK_SIZE, len(all_stocks))
        chunk_stocks = all_stocks[start_idx:end_idx]
        
        # 动态生成文件名，例如：08_fina_indicator_part1.xlsx
        file_name = f"{table_key}_part{chunk_idx + 1}.xlsx"
        file_path = OUT_DIR / file_name
        
        existing_df = pd.DataFrame()
        # 寻找断点
        if file_path.exists():
            try:
                existing_df = pd.read_excel(file_path)
                if not existing_df.empty and 'ts_code' in existing_df.columns:
                    done_stocks = existing_df['ts_code'].unique().tolist()
                    chunk_stocks = [s for s in chunk_stocks if s not in done_stocks]
            except Exception as e:
                logging.warning(f"⚠️ {file_name} 读取失败，将重建。")
        
        if not chunk_stocks:
            logging.info(f"☕ {file_name} 已是最新。")
            continue
            
        new_data = []
        logging.info(f"🚀 开始处理 {file_name} (剩余 {len(chunk_stocks)} 只股票)...")
        
        for i, ts_code in enumerate(chunk_stocks):
            try:
                df = getattr(pro, api_name)(ts_code=ts_code, fields=fields)
                if not df.empty: 
                    new_data.append(df)
                
                # 每 100 只股票保存一次，防断电/防网络中断
                if (i + 1) % 100 == 0:
                    temp_df = pd.concat([existing_df] + new_data, ignore_index=True)
                    temp_df.to_excel(file_path, index=False)
                    logging.info(f"💾 {file_name} 进度: {i + 1}/{len(chunk_stocks)} 只...")
                    
                time.sleep(SLEEP_TIME)
            except Exception as e:
                logging.error(f"❌ {table_key} 股票 {ts_code} 发生错误: {e}")
                break # 遇到错误跳出，等待下次续传
                
        # 最终合并保存当前 chunk
        if new_data:
            final_df = pd.concat([existing_df] + new_data, ignore_index=True)
            final_df.to_excel(file_path, index=False)
            logging.info(f"✅ {file_name} 同步完成！")

In [75]:
# --- Cell 5: 主控调度 ---
if __name__ == "__main__":
    # ==========================================
    # 🎯 运行开关：在这里控制你想拉取的表格
    # ==========================================
    # 如果想单独拉取某几个，把需要的留下，不需要的在前面加 # 注释掉即可：
    TARGET_TABLES = [
        # --- P0 基础与日频行情 ---
        # '01_stock_basic',
        # '02_trade_cal',
        # '03_daily',
        # '04_adj_factor',
        # '05_index_daily',
        # '06_daily_basic',

        # --- P1 财务报表 (by_stock模式) ---
        # '07_income',
        # '08_fina_indicator',
        # '09_forecast',
        # '10_express',

        # --- P2 增强与局部 ---
        #'11_disclosure_date',   
        #'12_suspend_d',
        #'13_stk_limit',
        #'14_sw_member',
        '15_moneyflow',
        '16_margin',
        '17_hk_hold',
        '18_stk_holdernumber',
        '19_top10_holders',
        '20_block_trade',
        '21_index_weight',
        '22_macro'
    ]

    # 💡 快捷键：如果您想一键拉取【所有】在 config.ipynb 里定义过的表格，
    # 只需取消下面这行代码的注释，它会无视上面的列表：
    # TARGET_TABLES = list(API_CONFIG.keys())

    # ==========================================

    logging.info(f"🏁 开始执行指定接口同步，共 {len(TARGET_TABLES)} 个任务...")
    
    for tid in TARGET_TABLES:
        if tid not in API_CONFIG:
            logging.warning(f"⚠️ 找不到 {tid} 的配置，已跳过。")
            continue
            
        cfg = API_CONFIG[tid]
        mode = cfg.get('split_mode')
        
        if mode is None:
            sync_full_table(tid, cfg['api'], cfg['fields'])
        elif mode == 'by_stock':
            sync_by_stock_logic(tid, cfg['api'], cfg['fields'])
        else:
            sync_dynamic_incremental(
                tid, 
                cfg['api'], 
                cfg['date_param'], 
                cfg['fields'], 
                mode, 
                cfg.get('extra_params')
            )
            
    logging.info("✨ 所有指定的同步任务已圆满结束。")

2026-05-15 12:11:37,183 - [INFO] - 🏁 开始执行指定接口同步，共 8 个任务...
2026-05-15 12:11:37,185 - [INFO] - 🚀 启动日期增量拉取: 15_moneyflow
2026-05-15 12:13:02,970 - [INFO] - 🚀 启动日期增量拉取: 16_margin
2026-05-15 12:13:04,826 - [INFO] - 🚀 启动日期增量拉取: 17_hk_hold
2026-05-15 12:13:35,335 - [INFO] - 🔄 启动按股票循环拉取: 18_stk_holdernumber
2026-05-15 12:13:55,477 - [INFO] - 🔄 启动按股票循环拉取: 19_top10_holders
2026-05-15 12:16:07,119 - [ERROR] - ❌ 19_top10_holders 股票 300708.SZ 失败: This sheet is too large! Your sheet size is: 1055622, 6 Max sheet size is: 1048576, 16384


ValueError: This sheet is too large! Your sheet size is: 1055622, 6 Max sheet size is: 1048576, 16384